In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown

here = Path.cwd().resolve()
for candidate in (here, *here.parents):
    if (candidate / "pyproject.toml").is_file():
        REPO = candidate
        break
else:
    raise RuntimeError("pstrain repository not found")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
display(Markdown("""# From waveform to words: HMM/GMM acoustic-model training
This tutorial turns CMU Arctic speech into a deployable `pstrain`/PocketSphinx model. You will inspect speech, prepare a pronunciation-aware corpus, train a context-dependent HMM/GMM, align phones, decode held-out audio, and test the package.

**WAV → MFCC → HMM/GMM → alignment → LM-assisted decoding → package**"""))
print("Repository:", REPO)

## Roadmap and runtime
The default uses 150 in-vocabulary SLT utterances and `cd-1g` (about 8–15 seconds on a many-core laptop). Other cells are normally under five seconds. The mtime DAG skips current work on reruns. Without the cached full corpus, the notebook uses the bundled 10-utterance corpus and `ci-1g`. For a larger experiment, increase the subset and choose `cd-8g`.

In [ ]:
%pip install -q matplotlib

import matplotlib
import numpy as np
import pstrain

assert pstrain.__version__ == "0.2.0"
print(
    f"numpy {np.__version__} | matplotlib {matplotlib.__version__} | "
    f"pstrain {pstrain.__version__}"
)

In [ ]:
import hashlib
import re
import shutil
import tarfile
import time
import urllib.request
import wave

import matplotlib.pyplot as plt

WORK = REPO / "notebooks/_work"
WORK.mkdir(parents=True, exist_ok=True)

VOICE = "slt"
N_UTTS = 150
TARGET = "cd-1g"
# Serial inside the notebook: parallel workers (jobs > 1) use a spawn-based
# ProcessPool that can deadlock when driven from a Jupyter kernel. From a
# terminal you can parallelize with `pstrain build ... -j N`.
JOBS = 1
OFFLINE = False
FORCE = False

np.random.seed(42)
print("Work directory:", WORK)
print("Utterances:", N_UTTS, "| target:", TARGET, "| jobs:", JOBS)

In [ ]:
from importlib.metadata import version

import pocketsphinx

DICT_FULL = REPO / "benchmarks/arctic/data/cmu_arctic_slt.dict"
MINI = REPO / "tests/fixtures/mini_arctic"

assert DICT_FULL.is_file()
assert (MINI / "wav").is_dir()
print("pstrain source:", Path(pstrain.__file__).resolve())
print("PocketSphinx version:", version("pocketsphinx"))

# Part II — Speech as a signal
A microphone samples air-pressure changes. Arctic WAVs store one 16-bit mono sample 16,000 times per second. Recognition uses overlapping frames because speech is approximately stationary for about 25 ms; a 10 ms step preserves timing.

In [ ]:
def read_wav(path):
    with wave.open(str(path), "rb") as w:
        meta = (w.getframerate(), w.getnchannels(), w.getsampwidth())
        x = (
            np.frombuffer(w.readframes(w.getnframes()), dtype="<i2").astype(float)
            / 32768
        )
    return x, meta


x, (sr, ch, sw) = read_wav(MINI / "wav/arctic_a0001.wav")
t = np.arange(len(x)) / sr
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(t, x, lw=0.5)
ax[0].set(title="Whole utterance", xlabel="time (s)", ylabel="amplitude")
a = int(0.35 * sr)
z = x[a : a + int(0.025 * sr)]
ax[1].plot(np.arange(len(z)) / sr * 1000, z)
ax[1].set(title="A 25 ms frame", xlabel="time (ms)")
plt.tight_layout()
plt.show()
print(sr, ch, sw * 8, len(x) / sr)

## Time becomes frequency
The short-time Fourier transform applies a Hann window, takes a 512-point real FFT, and advances 160 samples. Stacked spectra show when frequency energy occurs.

In [ ]:
def stft_power(x, sr=16000, nfft=512, win_s=0.025625, hop_s=0.01):
    nwin, hop = round(win_s * sr), round(hop_s * sr)
    starts = np.arange(0, max(1, len(x) - nwin + 1), hop)
    frames = np.stack([x[s : s + nwin] for s in starts]) * np.hanning(nwin)
    return (
        abs(np.fft.rfft(frames, n=nfft)) ** 2,
        starts / sr,
        np.fft.rfftfreq(nfft, 1 / sr),
    )


P, ft, fq = stft_power(x, sr)
plt.figure(figsize=(11, 4))
plt.pcolormesh(ft, fq / 1000, 10 * np.log10(P.T + 1e-10), shading="auto", cmap="magma")
plt.ylim(0, 8)
plt.xlabel("time (s)")
plt.ylabel("frequency (kHz)")
plt.title("NumPy-native STFT spectrogram")
plt.colorbar(label="dB")
plt.show()

## From spectra to MFCCs
The real front end uses pre-emphasis 0.97 → 25.6 ms frames/10 ms step → FFT512 → 25 mel filters (130–6800 Hz) → log → DCT13 → lifter22 → batch CMN → deltas and delta-deltas (39 dimensions). The next visualization stops at log-mel; production MFCCs run in pstrain's C feature code.

In [ ]:
def h2m(h):
    return 2595 * np.log10(1 + h / 700)


def m2h(m):
    return 700 * (10 ** (m / 2595) - 1)


def melbank(sr=16000, nfft=512, nf=25, lo=130, hi=6800):
    bins = np.floor(
        (nfft + 1) * m2h(np.linspace(h2m(lo), h2m(hi), nf + 2)) / sr
    ).astype(int)
    B = np.zeros((nf, nfft // 2 + 1))
    for i, (a, b, c) in enumerate(zip(bins[:-2], bins[1:-1], bins[2:])):
        B[i, a:b] = np.arange(b - a) / max(1, b - a)
        B[i, b:c] = np.arange(c - b, 0, -1) / max(1, c - b)
    return B


B = melbank(sr)
logmel = np.log(np.maximum(P @ B.T, 1e-12)).T
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(fq, B.T)
ax[0].set(xlim=(0, 8000), title="Mel filterbank", xlabel="Hz")
ax[1].imshow(
    logmel, origin="lower", aspect="auto", extent=[ft[0], ft[-1], 1, 25], cmap="magma"
)
ax[1].set(title="Illustrative log-mel energies", xlabel="time (s)")
plt.show()

# Part III — Corpus to project
CMU Arctic has phonetically balanced prompts from single speakers. SLT is a US English female voice with 1,132 WAVs and `txt.done.data`. Fetch order is WORK copy, local benchmark cache, download, then bundled mini-corpus.

In [ ]:
URL = "http://festvox.org/cmu_arctic/packed/cmu_us_slt_arctic.tar.bz2"
SHA = "7c173297916acf3cc7fcab2713be4c60b27312316765a90934651d367226b4ea"
arc = WORK / "cmu_us_slt_arctic.tar.bz2"
cache = Path.home() / ".cache/pstrain/benchmarks/cmu_us_slt_arctic.tar.bz2"
root = WORK / "corpus"
full = root / "cmu_us_slt_arctic"
mode = "WORK copy"
try:
    if not arc.exists():
        if cache.exists():
            shutil.copy2(cache, arc)
            mode = "benchmark cache"
        elif not OFFLINE:
            urllib.request.urlretrieve(URL, arc)
            mode = "download"
        else:
            raise FileNotFoundError()
    if hashlib.sha256(arc.read_bytes()).hexdigest() != SHA:
        raise ValueError("SHA-256 mismatch")
    if not full.exists():
        root.mkdir(parents=True, exist_ok=True)
        tarfile.open(arc, "r:bz2").extractall(root, filter="data")
    ARCTIC, WAV_DIR, DICT, IS_FULL = full, full / "wav", DICT_FULL, True
except (OSError, ValueError, tarfile.TarError) as e:
    ARCTIC, WAV_DIR, DICT, IS_FULL = MINI, MINI / "wav", MINI / "dictionary.dict", False
    mode = f"mini fallback: {type(e).__name__}"
print(mode, len(list(WAV_DIR.glob("*.wav"))))

## Words need pronunciations
Phonemes are contrastive sound categories; ARPABET writes them with ASCII: `cat → K AE T`. SLT uses 39 stress-stripped English phones plus `SIL`. Variants use suffixes such as `a(2)`. The dictionary maps words to phones; it is not a language model.

In [ ]:
def lex(path):
    d = {}
    for line in path.read_text().splitlines():
        if line.strip() and not line.startswith("#"):
            w, *ph = line.split()
            d[w] = ph
    return d


L = lex(DICT)
if IS_FULL:
    rr = re.compile(r'\(\s*(\S+)\s+"(.*)"\s*\)')
    prompts = {
        m.group(1): m.group(2)
        for line in (ARCTIC / "etc/txt.done.data").read_text().splitlines()
        if (m := rr.match(line.strip()))
    }
else:
    prompts = {}
    for line in (MINI / "transcription.txt").read_text().splitlines():
        u, t = line.split(maxsplit=1)
        prompts[u] = t
phones = sorted({p for v in L.values() for p in v} | {"SIL"})
variant = next((w for w in L if "(" in w), "none")
print(variant, L.get(variant))
print(len(phones), "phones:", " ".join(phones))

### One word, several pronunciation paths

A word can have several pronunciations. The lexicon writes them as `word`, `word(2)`, and `word(3)`; the inspection above showed one such variant.

With the pstrain default `training.multipron_training = True`, the utterance HMM graph includes **all** variants as parallel paths. Baum–Welch sums state and occupancy posteriors across those paths, so training learns from whichever pronunciation best matches the audio—there is no hard pre-selection. The alternative `linear` behavior selects the first observed pronunciation and is bit-identical to stock SphinxTrain.

```text
                       ┌─ DH  AH ─┐
... previous word ─────┤          ├───── next word ...
                       └─ DH  IY ─┘
                            “the”
```

This matters for speaker and dialect variation and for reduced function words such as “the” (`DH AH` versus `DH IY`). Sharing evidence across plausible paths improves acoustic estimates and forced alignment.

## Normalize and filter OOV prompts
Simple transcriptions are `fileid words…`; Sphinx form is `<s> words </s> (fileid)`. Case must match the lowercase lexicon. We lowercase, remove punctuation except internal apostrophes, trim edge apostrophes, and reject any prompt containing an OOV. Fillers `<s>`, `</s>`, and `<sil>` map to `SIL`.

In [ ]:
def norm(t):
    t = re.sub(r"[^\w\s']", " ", t.lower())
    return " ".join(q.strip("'") for q in t.split() if q.strip("'"))


normalized = {u: norm(t) for u, t in prompts.items()}
inv = {
    u: t
    for u, t in normalized.items()
    if t and all(w in L for w in t.split()) and (WAV_DIR / f"{u}.wav").exists()
}
selected = dict(list(sorted(inv.items()))[: N_UTTS if IS_FULL else len(inv)])
TARGET_RUN = TARGET if IS_FULL and len(selected) >= 100 else "ci-1g"
TRANS = WORK / "all.transcription"
TRANS.write_text("".join(f"{u} {t}\n" for u, t in selected.items()))
print(len(prompts), len(inv), len(selected), TARGET_RUN)
fig, axs = plt.subplots(min(3, len(selected)), 1, figsize=(11, 6), squeeze=False)
for ax, (u, t) in zip(axs[:, 0], list(selected.items())[:3]):
    y, (r, _, _) = read_wav(WAV_DIR / f"{u}.wav")
    q, tt, ff = stft_power(y, r)
    ax.pcolormesh(
        tt, ff / 1000, 10 * np.log10(q.T + 1e-10), shading="auto", cmap="magma"
    )
    ax.set_ylim(0, 8)
    ax.set_title(f"{u}: {t}", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
bad = []
for u, t in selected.items():
    with wave.open(str(WAV_DIR / f"{u}.wav"), "rb") as w:
        if (w.getframerate(), w.getnchannels(), w.getsampwidth()) != (16000, 1, 2):
            bad.append(u)
    if any(q not in L for q in t.split()):
        bad.append(u)
assert not bad
print(f"✓ {len(selected)} WAVs are 16 kHz/mono/16-bit; all words are in-vocabulary")

In [ ]:
from pstrain.api import setup_project

PROJECT = WORK / "slt_project"
setup = setup_project(PROJECT, TRANS, WAV_DIR, DICT, link_audio=True, clobber=False)
print("project", PROJECT)
print(setup)
for q in sorted(PROJECT.rglob("*")):
    if q.is_file() or q.is_symlink():
        print(q.relative_to(PROJECT))

In [ ]:
from pstrain.api.pipeline import PipelineContext, build_pipeline
from pstrain.api import parse_transcription_file

ctx = PipelineContext.from_config(
    PROJECT,
    experiment="default",
    config_name="default",
    cli_overrides={"runner": {"jobs": JOBS}},
)
pipeline = build_pipeline(ctx)
assert pipeline.run("split", jobs=JOBS) == 0
ETC = PROJECT / "experiments/default/etc"
train_transcripts = parse_transcription_file(ETC / "train.transcription")
test_transcripts = parse_transcription_file(ETC / "test.transcription")
plt.bar(["train", "test"], [len(train_transcripts), len(test_transcripts)])
plt.ylabel("utterances")
plt.title("Split (seed 42)")
plt.show()
print("train:", len(train_transcripts), "| test:", len(test_transcripts))

# Part IV — Features and configuration
The production chain ends in normalized 39-dimensional cepstra. `feat.params` persists the entire front-end contract beside the model, so decoding reproduces training features.

In [ ]:
from pstrain.lib.config import resolve_config

cfg = resolve_config(PROJECT, profile_name="default").as_dict()
feature_fields = [
    "samprate",
    "ncep",
    "nfilt",
    "nfft",
    "frate",
    "wlen",
    "feat_type",
    "alpha",
    "lifter",
]
for field in feature_fields:
    print(f"features.{field}: {cfg['features'][field]}")

for field in ("n_state", "n_senones"):
    print(f"training.{field}: {cfg['training'][field]}")

for family in ("ci", "untied", "tied"):
    print(f"training.{family}: {cfg['training'][family]}")

In [ ]:
assert pipeline.run("features", jobs=JOBS) == 0
fp = PROJECT / "shared/features/default/feat.params"
print(fp.read_text())
dct = np.cos(np.pi / 25 * (np.arange(25) + 0.5) * np.arange(13)[:, None])
cep = (dct @ logmel).T
cep -= cep.mean(0)
plt.imshow(cep.T, origin="lower", aspect="auto", cmap="coolwarm")
plt.title("Illustrative NumPy cepstra; pstrain extracts production features")
plt.xlabel("frame")
plt.ylabel("coefficient")
plt.show()

# Part V — The model
Each phone is a three-emitting-state left-to-right HMM: self-loop 0.75, forward 0.25. Each state scores 39-D frames with a diagonal Gaussian mixture. The HMM models time; the GMM models feature regions. $\log N(x;\mu,\sigma^2)=\log(1/\sqrt{2\pi|\Sigma|})-rac12\sum_d(x_d-\mu_d)^2/\sigma_d^2$.

In [ ]:
np.random.seed(0)

points = np.r_[
    np.random.randn(80, 2) * [0.5, 0.8] + [-1, 0],
    np.random.randn(70, 2) * [0.6, 0.4] + [1.2, 1],
]
means = np.array([[-1, 0], [1.2, 1]])
variances = np.array([[0.25, 0.64], [0.36, 0.16]])


def diagonal_gaussian_pdf(values, mean, variance):
    exponent = -0.5 * np.sum((values - mean) ** 2 / variance, axis=1)
    normalizer = np.sqrt((2 * np.pi) ** 2 * np.prod(variance))
    return np.exp(exponent) / normalizer


# Normalize weighted likelihoods to obtain posterior component responsibilities.
weighted_likelihoods = np.c_[
    0.55 * diagonal_gaussian_pdf(points, means[0], variances[0]),
    0.45 * diagonal_gaussian_pdf(points, means[1], variances[1]),
]
responsibilities = weighted_likelihoods / weighted_likelihoods.sum(
    axis=1, keepdims=True
)

plt.scatter(*points.T, c=responsibilities[:, 1], cmap="coolwarm")
plt.scatter(*means.T, color="black", marker="x", s=100)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Diagonal-GMM soft assignments")
plt.colorbar(label="responsibility for component 2")
plt.show()

## Model families
`ci-1g…ci-8g` are monophones. `cd-untied` gives each observed triphone its own states and is an intermediate. `cd-1g…cd-32g` use decision-tree-tied triphone states. A phone is a sound, an HMM state a temporal phase, a senone a shared tied-state distribution, and a Gaussian one component. Density is selected by target name; `cd-8g` is the toolkit default and `cd-1g` is this fast full-CD path.

In [ ]:
assert pipeline.run(TARGET_RUN, dry_run=True, jobs=JOBS, verbose=True) == 0
st = ["features", "split", "flat", "ci-1g"] + (
    []
    if TARGET_RUN == "ci-1g"
    else ["cd-untied", "questions", "trees", "prune", "cd-1g"]
)
xx = np.arange(len(st))
plt.plot(xx, np.zeros(len(st)))
plt.scatter(xx, np.zeros(len(st)), s=600)
for index, stage in enumerate(st):
    plt.text(index, 0, stage, ha="center", va="center", fontsize=8)
plt.axis("off")
plt.title("Target dependency path")
plt.show()

In [ ]:
# HEAVY CELL: real pstrain training DAG
ctx = PipelineContext.from_config(
    PROJECT,
    experiment="default",
    config_name="default",
    cli_overrides={"runner": {"jobs": JOBS}},
)
pipeline = build_pipeline(ctx)
started = time.perf_counter()
rc = pipeline.run(TARGET_RUN, force=FORCE, jobs=JOBS)
TRAIN_SECONDS = time.perf_counter() - started
assert rc == 0
MODEL_DIR = PROJECT / "shared/models" / TARGET_RUN / "default"
CI_MODEL_DIR = PROJECT / "shared/models/ci-1g/default"
print(f"TRAIN_WALL_SECONDS={TRAIN_SECONDS:.3f}")
print(MODEL_DIR)
print(f"CLI: pstrain build {TARGET_RUN} --project-dir {PROJECT} -j 4 -c default")

In [ ]:
required = [
    "mdef",
    "means",
    "variances",
    "mixture_weights",
    "transition_matrices",
    "feat.params",
]
for f in required:
    q = MODEL_DIR / f
    assert q.is_file()
    print(f, q.stat().st_size)
rows = [
    z.split()
    for z in (MODEL_DIR / "mdef").read_text(errors="replace").splitlines()
    if len(z.split()) >= 10 and z.split()[-1] == "N" and not z.startswith("#")
]
print("base lft rt pos attrib tmat s0 s1 s2 N")
for row in rows[:4]:
    print(" ".join(row))
if TARGET_RUN.startswith("cd-"):
    ids = [s for r in rows for s in r[6:9]]
    print(len(ids), "state references", len(set(ids)), "unique senone IDs")

# Part VI — What training did
**Flat initialization:** pstrain estimates one global Gaussian over all frames and tiles its mean and variance into every phone state. Topology exists, but states are not acoustically specialized.

In [ ]:
np.random.seed(0)
toy = np.r_[np.random.normal(-0.5, 1, (100, 2)), np.random.normal(1, 0.6, (80, 2))]
gm = toy.mean(0)
gv = toy.var(0)
sm = np.tile(gm, (3, 1))
plt.scatter(*toy.T, s=10, alpha=0.3)
plt.scatter(
    sm[:, 0], sm[:, 1], s=[80, 160, 240], facecolors="none", edgecolors=["r", "g", "b"]
)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Flat init: every state shares one Gaussian")
plt.show()
print(gm, gv)

In [ ]:
obs = np.array([-1.1, -0.7, -0.2, 0.4, 0.9, 1.2])
mu = np.array([-0.8, 0.7])
var = np.ones(2) * 0.5
A = np.array([[0.75, 0.25], [0, 1.0]])


def fb():
    B = np.exp(-0.5 * (obs[:, None] - mu) ** 2 / var) / np.sqrt(2 * np.pi * var)
    a = np.zeros_like(B)
    a[0] = [1, 0] * B[0]
    sc = []
    for t in range(len(obs)):
        if t:
            a[t] = (a[t - 1] @ A) * B[t]
        sc.append(a[t].sum())
        a[t] /= max(sc[-1], 1e-300)
    bt = np.ones_like(B)
    for t in range(len(obs) - 2, -1, -1):
        bt[t] = A @ (B[t + 1] * bt[t + 1])
        bt[t] /= max(bt[t].sum(), 1e-300)
    g = a * bt
    g /= g.sum(1, keepdims=True)
    return g, np.log(sc).sum()


g, ll0 = fb()
old = mu.copy()
mu = (g * obs[:, None]).sum(0) / g.sum(0)
g, ll1 = fb()
plt.imshow(g.T, aspect="auto", cmap="Blues")
plt.yticks([0, 1], ["state 1", "state 2"])
plt.title("Baum–Welch posterior occupancy γ")
plt.show()
print(old, "→", mu, ll0, "→", ll1)
print(
    "mean=macc/dnom; var=vacc/dnom−mean²; floor 1e-4; stop at signed per-frame ΔLL ≤ .001; CI/untied/tied max 10/6/10"
)

In [ ]:
m = np.array([0.0, 0.0])
v = np.array([1.0, 0.25])
kids = np.stack([m - 0.2 * np.sqrt(v), m + 0.2 * np.sqrt(v)])
plt.scatter([0], [0], s=180, label="parent")
plt.scatter(kids[:, 0], kids[:, 1], s=130, label="children")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.legend()
plt.title("Split μ ± 0.2σ; halve weight; copy variance")
plt.show()
print(kids, "weights .5/.5")

## Context and tying
A monophone becomes many triphones because neighboring phones affect acoustics. pstrain trains reachable untied triphones, clusters CI phones into phonetic questions, builds one tree per `(phone,state)`, globally prunes lowest-gain twigs to the senone budget, and maps contexts to surviving leaves.

In [ ]:
contexts = np.array(["R", "L", "S", "K", "W", "N"])
occ = np.array([50, 35, 60, 40, 25, 45])
mm = np.array([1, 0.9, -0.7, -0.5, 1.2, -0.3])
vv = np.array([0.4, 0.5, 0.6, 0.5, 0.3, 0.7])
Q = {
    "liquids/glides": np.isin(contexts, ["R", "L", "W"]),
    "coronal/nasal": np.isin(contexts, ["S", "N"]),
}


def gain(mask):
    def cost(ix):
        w = occ[ix]
        m = (w * mm[ix]).sum() / w.sum()
        v = (w * (vv[ix] + (mm[ix] - m) ** 2)).sum() / w.sum()
        return 0.5 * w.sum() * np.log(v)

    return cost(np.ones(len(occ), bool)) - cost(mask) - cost(~mask)


scores = {k: gain(v) for k, v in Q.items()}
best = max(scores, key=scores.get)
plt.bar(scores.keys(), scores.values())
plt.ylabel("likelihood gain")
plt.title("Greedy question split: " + best)
plt.show()
print(scores)

# Part VII — Alignment and decoding
Baum–Welch uses forward–backward during training and assigns fractional occupancy. Forced alignment uses Viterbi with known words. Decoding uses Viterbi beam search with unknown words plus a lexicon and LM.

### Why HMM/GMM still matters

Newer acoustic models—DNN-HMM hybrids, end-to-end CTC and transducer systems, self-supervised models such as wav2vec 2.0, and Whisper—achieve lower word error rates on many benchmarks. HMM/GMM nevertheless remains strong and widely used for **forced alignment** and temporal localization: phone and word time stamps. It is the backbone of major forced aligners including Montreal Forced Aligner, Prosodylab-Aligner, and Gentle.

Alignment constrains search to a known transcript: a small possibility lattice rather than every possible sentence. That constraint lets a simple, fast, CPU-only, deterministic, interpretable acoustic model localize phones accurately—often well enough and more cheaply than a large model. The alignment-overlay figure below is exactly this use case.

In [ ]:
from pstrain.api.alignment import align_corpus, save_ctm, save_textgrid

# A handful of utterances is enough to find a clean example for the overlay.
alignment_subset = dict(list(train_transcripts.items())[:6])
job = align_corpus(
    alignment_subset,
    PROJECT / "audio",
    CI_MODEL_DIR,
    PROJECT / "shared/dictionary.dict",
    PROJECT / "shared/filler.dict",
    include_phones=True,
)
assert job.n_aligned > 0, job.errors

aligned_uid, alignment = next(iter(job.results.items()))
print(job.n_aligned, job.n_failed, aligned_uid, alignment.duration_time())
for segment in alignment.phones[:12]:
    print(
        segment.name,
        segment.start_time(alignment.frame_shift),
        segment.end_time(alignment.frame_shift),
    )

In [ ]:
aligned_waveform, (aligned_sample_rate, _, _) = read_wav(
    PROJECT / "audio" / f"{aligned_uid}.wav"
)
aligned_power, aligned_times, aligned_frequencies = stft_power(
    aligned_waveform, aligned_sample_rate
)
fig, ax = plt.subplots(figsize=(13, 4))
ax.pcolormesh(
    aligned_times,
    aligned_frequencies / 1000,
    10 * np.log10(aligned_power.T + 1e-10),
    shading="auto",
    cmap="magma",
)
ax.set_ylim(0, 8)
for segment in alignment.phones:
    start = segment.start_time(alignment.frame_shift)
    end = segment.end_time(alignment.frame_shift)
    color = "cyan" if segment.name == "SIL" else "white"
    ax.axvspan(start, end, color=color, alpha=0.07)
    ax.axvline(start, color="white", lw=0.4)
    ax.text(
        (start + end) / 2,
        7.5,
        segment.name,
        rotation=90,
        ha="center",
        va="top",
        fontsize=7,
        color="white",
    )
ax.set(title=f"CI-1G phone alignment — {aligned_uid}", xlabel="time (s)", ylabel="kHz")
plt.show()

In [ ]:
CTM = WORK / f"{aligned_uid}.phones.ctm"
TG = WORK / f"{aligned_uid}.TextGrid"
save_ctm(alignment, CTM, level="phones")
save_textgrid(alignment, TG)
print("\n".join(CTM.read_text().splitlines()[:8]))
print("\nTextGrid:\n" + "\n".join(TG.read_text().splitlines()[:8]))

## Decoder ingredients
PocketSphinx loads means, variances, mixture weights, transition matrices, `mdef`, `feat.params`, and optional `sendump`. Forward-tree → forward-flat → best-path beam search combines GMM/HMM acoustics, dictionary pronunciations, and an ARPA n-gram. Defaults include beam `1e-80`, word beam `1e-40`, LM weight 10, and insertion penalty 0.2.

In [ ]:
from pstrain.lib.lm import build_lm

LM = build_lm(list(train_transcripts.values()), WORK / "train.arpa", 3)
assert Path(LM).is_file()
print(LM, Path(LM).stat().st_size, "bytes")
print("The LM is built directly; decoding always receives an explicit LM.")

In [ ]:
from pstrain.api.testing import test_model

decode = test_model(
    MODEL_DIR,
    PROJECT / "audio",
    test_transcripts,
    PROJECT / "shared/dictionary.dict",
    PROJECT / "shared/filler.dict",
    lm=LM,
    verbose=True,
    jobs=JOBS,
)
assert decode.n_decoded == len(test_transcripts)
DECODE_WER = decode.wer
print(f"decoded={decode.n_decoded}/{decode.n_utterances} WER={DECODE_WER:.3f}")
for u, z in list(decode.per_utterance.items())[:3]:
    print(u, "\n REF:", z["reference"], "\n HYP:", z["hypothesis"])
print(
    "High WER is expected from a tiny 1-Gaussian model and tiny LM; this is a green learning spine, not a benchmark."
)

# Part VIII — Settings change the experiment
Profiles: `default`, `wideband`, `telephone`, `wideband_large`, `sphinxtrain`. Precedence is built-in < user < project < experiment < CLI. Inspect with `resolve_config(...).as_dict()` or `pstrain config show/get/explain/list/schema`.

### What the model identifies is your choice

The pipeline is unit-agnostic: you supply the phoneset, lexicon, transcripts, and audio, and the model learns the units to which the lexicon maps words. That gives wide latitude:

- **Phonemes:** the units used in this notebook.
- **Stress or no stress:** CMUdict marks vowel stress with digits such as `AH0`, `AH1`, and `AH2`. Keeping them makes each stressed vowel a distinct phone: more context detail, a larger inventory, and a greater data requirement. This notebook's dictionary strips stress for a smaller inventory.
- **Visemes:** mouth-shape classes form a reduced-cardinality unit set for lip-reading or talking-head animation.
- **Another language:** swap the phoneset, lexicon, and training data, then choose a matching configuration profile—for example, the appropriate sample rate. The training machinery is unchanged.

To build a viseme model, map phones to visemes, rewrite every dictionary pronunciation as a viseme sequence, and derive the phoneset from that rewritten lexicon for validation. Then run the **same pipeline**. Fewer units provide more training examples per unit, so decision-tree training is easier and faster than for the phoneme model; recall that rare-phone tree failures occur when a unit has too little data.

In [ ]:
# One simplified mapping; real inventories vary (for example, Disney 12,
# Jeffers–Barley, and MPEG-4).
VISEME = {
    "SIL": "sil",
    "P": "PBM", "B": "PBM", "M": "PBM",
    "F": "FV", "V": "FV",
    "TH": "TH", "DH": "TH",
    "T": "TDNL", "D": "TDNL", "N": "TDNL", "L": "TDNL",
    "S": "SZ", "Z": "SZ",
    "SH": "SH", "ZH": "SH", "CH": "SH", "JH": "SH",
    "K": "KG", "G": "KG", "NG": "KG", "HH": "KG",
    "R": "R", "W": "W", "Y": "Y",
    "IY": "IY", "IH": "IY",
    "EH": "EH", "EY": "EH", "AE": "EH",
    "AA": "AA", "AH": "AA", "AO": "AA",
    "UW": "UW", "UH": "UW", "OW": "UW",
    "AW": "AY", "AY": "AY", "OY": "AY",
    "ER": "ER",
}


def collapse_repeats(units):
    """Collapse adjacent identical units, as many real viseme lexicons do."""
    collapsed = []
    for unit in units:
        if not collapsed or unit != collapsed[-1]:
            collapsed.append(unit)
    return collapsed


# Transform every pronunciation, including word(2), word(3), and later variants.
phoneme_lexicon = lex(DICT_FULL)
viseme_lexicon = {
    word: collapse_repeats([VISEME[phone] for phone in pronunciation])
    for word, pronunciation in phoneme_lexicon.items()
}
phoneme_inventory = sorted(
    {phone for pronunciation in phoneme_lexicon.values() for phone in pronunciation}
    | {"SIL"}
)
viseme_phoneset = sorted(
    {viseme for pronunciation in viseme_lexicon.values() for viseme in pronunciation}
    | {"sil"}
)

print("Phoneme inventory:", len(phoneme_inventory))
print("Viseme inventory:", len(viseme_phoneset))
print("Reduction:", len(phoneme_inventory) - len(viseme_phoneset), "units")
print("Viseme phoneset:", " ".join(viseme_phoneset))

for word in ("the", "author", "of"):
    phones = phoneme_lexicon[word]
    visemes = viseme_lexicon[word]
    print(f"{word:>6}: {' '.join(phones):<20} -> {' '.join(visemes)}")

print(
    "To train this, write the viseme lexicon (and optionally the viseme phoneset), "
    "point setup_project at them, and run the same pipeline."
)

In [ ]:
from pstrain.api.pipeline import TARGETS

for s in TARGETS:
    if s.name in ("cd-1g", "cd-8g", "cd-32g"):
        print(s.name, s.n_density, "Gaussian(s)/state")
plt.bar(["1g", "2g", "4g", "8g", "16g", "32g"], [1, 2, 4, 8, 16, 32])
plt.ylabel("densities/state")
plt.title("Target ladder: split, then retrain")
plt.show()

In [ ]:
large = resolve_config(PROJECT, profile_name="wideband_large").as_dict()
print(
    "default",
    cfg["training"]["n_senones"],
    "wideband_large",
    large["training"]["n_senones"],
)
preview = PipelineContext.from_config(
    PROJECT,
    experiment="senone-preview",
    config_name="default",
    cli_overrides={"training": {"n_senones": 100}},
)
assert build_pipeline(preview).run("cd-1g", dry_run=True, jobs=JOBS) == 0
print("Separate 100-senone experiment previewed; no retraining.")

In [ ]:
telephone_features = resolve_config(PROJECT, profile_name="telephone").as_dict()[
    "features"
]
default_features = cfg["features"]

print(f"{'setting':12s} {'default':>10s} {'telephone':>10s}")
for field in ("samprate", "nfilt", "nfft", "lowerf", "upperf"):
    print(
        f"{field:12s} {str(default_features[field]):>10s} "
        f"{str(telephone_features[field]):>10s}"
    )
print(
    "An 8 kHz channel trades bandwidth/resolution; changing features requires retraining."
)

## Capacity guide
CI is data-efficient; CD models neighboring-phone effects. Untied CD is sparse; tree-tied senones share evidence. More Gaussians model more modes but cost compute and risk overfitting. More senones preserve context detail but need data. CI/untied/tied schedules allow 10/6/10 iterations and stop at per-frame likelihood improvement ≤ .001.

# Part IX — Package and deploy
Deployment needs acoustic parameters and `feat.params`, main and filler dictionaries, and an explicit ARPA LM. Test the exact package you distribute.

In [ ]:
from pstrain.lib.steps.package import package_model

PROOT = WORK / "package"
packaged = package_model(
    MODEL_DIR,
    PROOT,
    model_name="arctic-tutorial",
    dictionary_path=PROJECT / "shared/dictionary.dict",
    filler_dict_path=PROJECT / "shared/filler.dict",
    include_dict=True,
)
PACKAGE = PROOT / "arctic-tutorial"
print(packaged)
for q in sorted(PACKAGE.rglob("*")):
    if q.is_file():
        print(q.relative_to(PACKAGE), q.stat().st_size)

In [ ]:
packaged_decode = test_model(
    PACKAGE / "acoustic",
    PROJECT / "audio",
    test_transcripts,
    PACKAGE / "dict/cmudict.dict",
    PACKAGE / "dict/filler.dict",
    lm=LM,
    verbose=True,
    jobs=JOBS,
)
assert packaged_decode.n_decoded == len(test_transcripts)
assert abs(packaged_decode.wer - DECODE_WER) < 1e-12
print(f"PACKAGED_SMOKE_WER={packaged_decode.wer:.3f}")
u, z = next(iter(packaged_decode.per_utterance.items()))
print(u, "\n REF:", z["reference"], "\n HYP:", z["hypothesis"])
print("✓ packaged model decodes successfully")

# Recap
Spectrograms expose time-frequency energy; MFCCs compress it. Phones are sounds, states are temporal phases, senones are shared distributions, and Gaussians are mixture components. CI ignores context; CD uses tied triphones. Baum–Welch trains; Viterbi aligns and decodes. Target names select family/density; config selects features, schedules, splits, and senones. Next: add in-vocabulary speech, try `cd-8g`, adjust senones only with enough data, train a larger LM, and rerun the packaged smoke test.

HMM/GMM remains especially valuable for fast, interpretable forced alignment when the transcript is known and accurate time stamps matter.